<a href="https://colab.research.google.com/github/Pramila003/OCRFinetunewithcustomeDataset/blob/main/detectionmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

!pip install -q datasets tqdm pillow

In [ ]:
import json
import os
from datasets import load_dataset
from tqdm import tqdm

# Set base output directory in Colab session
OUTPUT_DIR = "paddle_det_data"
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)

# 1. Download dataset from Hugging Face
print("Downloading FUNSD dataset from Hugging Face...")
dataset = load_dataset("jsdnrs/ICDAR2019-SROIE", split="train")

train_txt_path = os.path.join(OUTPUT_DIR, "train.txt")

# 2. Process records into PaddleOCR format
with open(train_txt_path, "w", encoding="utf-8") as f:
    for idx, item in enumerate(tqdm(dataset, desc="Converting FUNSD")):
        img = item["image"]
        width, height = img.size  # Get actual pixel dimensions

        img_filename = f"train_{idx:04d}.png"
        img_rel_path = f"images/{img_filename}"
        img_save_path = os.path.join(OUTPUT_DIR, img_rel_path)

        # Save image locally in Colab
        img.save(img_save_path)

        paddle_boxes = []
        words = item["words"]
        bboxes = item["bboxes"]

        for word, bbox in zip(words, bboxes):
            # Denormalize [xmin, ymin, xmax, ymax] from 0-1000 scale to pixel coordinates
            xmin = int(bbox[0] * width / 1000.0)
            ymin = int(bbox[1] * height / 1000.0)
            xmax = int(bbox[2] * width / 1000.0)
            ymax = int(bbox[3] * height / 1000.0)

            if xmax <= xmin or ymax <= ymin:
                continue

            # Convert to 4-point polygon format
            points = [
                [xmin, ymin],  # Top-Left
                [xmax, ymin],  # Top-Right
                [xmax, ymax],  # Bottom-Right
                [xmin, ymax],  # Bottom-Left
            ]

            paddle_boxes.append(
                {
                    "transcription": word if word.strip() else "###",
                    "points": points,
                }
            )

        if paddle_boxes:
            f.write(f"{img_rel_path}\t{json.dumps(paddle_boxes)}\n")

print(f"\nConversion complete! Folder generated at '/content/{OUTPUT_DIR}'")

README.md:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  319MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

Converting FUNSD: 100%|██████████| 626/626 [04:36<00:00,  2.26it/s]


Conversion complete! Folder generated at '/content/paddle_det_data'


In [ ]:
!cp -r paddle_det_data /content/drive/MyDrive/
print("Dataset backed up to Google Drive successfully!")

Dataset backed up to Google Drive successfully!


In [ ]:
# 1. Navigate to your project folder in Google Drive
%cd "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface"



/content/drive/MyDrive/Colab Notebooks/finetunehuggingface


In [ ]:
# 2. Create the directory if it doesn't exist
!mkdir -p pretrain_models



In [ ]:
!wget https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_train.tar -O pretrain_models/ch_PP-OCRv4_det_train.tar
!tar -xf pretrain_models/ch_PP-OCRv4_det_train.tar -C pretrain_models/

--2026-08-20 10:35:43--  https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_train.tar
Resolving paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14305280 (14M) [application/x-tar]
Saving to: ‘pretrain_models/ch_PP-OCRv4_det_train.tar’

pretrain_models/ch_ 100%[===================>]  13.64M  1.77MB/s    in 26s     

2026-08-20 10:36:11 (527 KB/s) - ‘pretrain_models/ch_PP-OCRv4_det_train.tar’ saved [14305280/14305280]



In [ ]:
# 5. Verify extracted files
!ls -l pretrain_models/ch_PP-OCRv4_det_train

total 13963
-rw------- 1 root root 14298107 Aug 20 10:36 best_accuracy.pdparams


In [ ]:
import os
import yaml

base_dir = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface"

# 1. Base Detection config (PP-OCRv4 student model)
det_config_path = os.path.join(
    base_dir, "PaddleOCR/configs/det/PP-OCRv4/PP-OCRv4_mobile_det.yml"
)

with open(det_config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. Point directly to 'best_accuracy' inside your extracted folder
config["Global"]["pretrained_model"] = os.path.join(
    base_dir, "pretrain_models/ch_PP-OCRv4_det_train/best_accuracy"
)
config["Global"]["save_model_dir"] = os.path.join(
    base_dir, "output/det_finetuned"
)
config["Global"]["use_gpu"] = True
config["Global"]["epoch_num"] = 50

# 3. Dataset paths
config["Train"]["dataset"]["data_dir"] = base_dir
config["Train"]["dataset"]["label_file_list"] = [
    os.path.join(base_dir, "train.txt")
]

config["Eval"]["dataset"]["data_dir"] = base_dir
config["Eval"]["dataset"]["label_file_list"] = [
    os.path.join(base_dir, "val.txt")
]

# 4. Batch Size Configuration
# Batch size per GPU card for training (775 images / 8 = ~96 steps per epoch)
config["Train"]["loader"]["batch_size_per_card"] = 8
config["Train"]["loader"]["drop_last"] = True
config["Train"]["loader"]["num_workers"] = 2

# Batch size per GPU card for validation
config["Eval"]["loader"]["batch_size_per_card"] = 1
config["Eval"]["loader"]["drop_last"] = False
config["Eval"]["loader"]["num_workers"] = 1

# 4. Save custom detection config file
custom_config_path = os.path.join(
    base_dir, "PaddleOCR/configs/det/custom_det_config.yml"
)
with open(custom_config_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f)

print(f"Config successfully generated at:\n{custom_config_path}")

Config successfully generated at:
/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR/configs/det/custom_det_config.yml


In [ ]:
# Install PaddlePaddle GPU v2.6.2
# Correct Colab command for CUDA 11.2
!nvidia-smi



Tue Aug 25 04:56:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import paddle

print("Paddle Version:", paddle.__version__)
print("CUDA Available:", paddle.is_compiled_with_cuda())

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Paddle Version: 3.3.1
CUDA Available: True


In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR
!pip install -r requirements.txt

/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.7 MB/s eta 0:00:00


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR"
!python tools/train.py -c configs/det/custom_det_config.yml

/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/08/25 05:22:10] ppocr INFO: Architecture : 
[2026/08/25 05:22:11] ppocr INFO:     Backbone : 
[2026/08/25 05:22:11] ppocr INFO:         det : True
[2026/08/25 05:22:11] ppocr INFO:         name : PPLCNetV3
[2026/08/25 05:22:11] ppocr INFO:         scale : 0.75
[2026/08/25 05:22:11] ppocr INFO:     Head : 
[2026/08/25 05:22:11] ppocr INFO:         fix_nan : True
[2026/08/25 05:22:11] ppocr INFO:         k : 50
[2026/08/25 05:22:11] ppocr INFO:         name : DBHead
[2026/08/25 05:22:11] ppocr INFO:     Neck : 
[2026/08/25 05:22:11] ppocr INFO:         nam

In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR"

!python tools/export_model.py \
    -c "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR/configs/det/custom_det_config.yml" \
    -o Global.pretrained_model="/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/det_finetuned/best_accuracy" \
    -o Global.save_inference_dir="/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/det_inference"

/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
W0824 06:34:27.440800 29583 gpu_resources.cc:116] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.6
[2026/08/24 06:34:28] ppocr WARNING: The pretrained params backbone.conv1.hardswish.scale not in model
[2026/08/24 06:34:28] ppocr WARNING: The pretrained params backbone.conv1.hardswish.bias not in model
[2026/08/24 06:34:28] ppocr INFO: load pretrain successful from /content/drive/MyDrive/Colab Notebooks/finetunehuggingface/pretrain_models/ch_PP-OCRv4_det_train/best_accuracy
[2026/08/24 06:34:28] ppocr INF

In [ ]:
!ls -la "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/det_finetuned"

total 138487
-rw------- 1 root root     3582 Aug 24 05:26 config.yml
-rw------- 1 root root 28384438 Aug 24 05:57 iter_epoch_10.pdopt
-rw------- 1 root root 14381328 Aug 24 05:57 iter_epoch_10.pdparams
-rw------- 1 root root      119 Aug 24 05:57 iter_epoch_10.states
-rw------- 1 root root 28384438 Aug 24 06:29 iter_epoch_20.pdopt
-rw------- 1 root root 14381328 Aug 24 06:29 iter_epoch_20.pdparams
-rw------- 1 root root      119 Aug 24 06:29 iter_epoch_20.states
-rw------- 1 root root 28384438 Aug 24 06:29 latest.pdopt
-rw------- 1 root root 14381328 Aug 24 06:29 latest.pdparams
-rw------- 1 root root      102 Aug 24 06:29 latest.states
-rw------- 1 root root 13506372 Aug 24 06:29 train.log


In [ ]:
import sys, platform
print("Python Version:", sys.version)
print("Architecture:", platform.architecture()[0], platform.machine())

Python Version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Architecture: 64bit x86_64


In [ ]:
!pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 840.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

import zipfile
import os

# Path to your zip file
zip_path = '/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/rec_inference.zip'

# Destination folder where you want to extract
extract_path = '/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output'

# Make sure the destination exists
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipping complete! Files are in:", extract_path)



✅ Unzipping complete! Files are in: /content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output


In [ ]:
import sys
import os
import cv2
import yaml
import numpy as np
import logging

# 1. Silence PaddleOCR debug logs
logging.getLogger("ppocr").setLevel(logging.ERROR)
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

# 2. Add PaddleOCR to python path
repo_path = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR"
sys.path.append(repo_path)

import tools.infer.utility as utility
from tools.infer.predict_system import TextSystem

# 3. Ensure valid model names in inference.yml
det_yml_path = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/det_inference/inference.yml"
rec_yml_path = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/inference/inference.yml"

def set_model_name(yml_path, model_name_str):
    if os.path.exists(yml_path):
        try:
            with open(yml_path, "r") as f: data = yaml.safe_load(f)
            if not isinstance(data, dict): data = {}
            if "Global" not in data or not isinstance(data["Global"], dict): data["Global"] = {}
            data["Global"]["model_name"] = model_name_str
            with open(yml_path, "w") as f: yaml.dump(data, f)
        except Exception: pass

set_model_name(det_yml_path, "PP-OCRv5_mobile_det")
set_model_name(rec_yml_path, "PP-OCRv5_mobile_rec")

# 4. Setup arguments for direct TextSystem
sys.argv = ['']
args = utility.parse_args()

args.det_model_dir = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/det_inference"
args.rec_model_dir = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/inference"
args.rec_char_dict_path = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/inference/ppocr_keys.txt"
args.use_gpu = False
args.enable_mkldnn = False
args.ir_optim = False
args.use_space_char = True
args.use_angle_cls = False
args.det_db_unclip_ratio = 2.2 # Expands boxes so words on the same line merge cleanly
args.det_db_box_thresh = 0.3


# 5. Initialize direct TextSystem (Full-resolution OCR)
text_sys = TextSystem(args)

# 6. Dynamic Median Line-Height Sorting Algorithm
def get_perfect_reading_order(dt_boxes, rec_res):
    if dt_boxes is None or len(dt_boxes) == 0:
        return "No text detected."

    items = []
    for box, (text, score) in zip(dt_boxes, rec_res):
        pts = np.array(box)
        y_min = np.min(pts[:, 1])
        y_max = np.max(pts[:, 1])
        x_min = np.min(pts[:, 0])
        height = y_max - y_min
        y_center = (y_min + y_max) / 2.0

        items.append({
            'text': text,
            'score': score,
            'y_center': y_center,
            'x_min': x_min,
            'height': height
        })

    # Calculate median line height to dynamically group lines
    heights = [item['height'] for item in items]
    median_height = np.median(heights) if len(heights) > 0 else 20.0
    line_threshold = median_height * 0.6  # 60% of average line height

    # Sort all detected boxes top-to-bottom by Y-center
    items = sorted(items, key=lambda x: x['y_center'])

    # Group boxes into horizontal lines dynamically
    lines = []
    for item in items:
        placed = False
        for line in lines:
            line_y_center = np.mean([it['y_center'] for it in line])
            if abs(item['y_center'] - line_y_center) < line_threshold:
                line.append(item)
                placed = True
                break
        if not placed:
            lines.append([item])

    # Sort lines top-to-bottom
    lines = sorted(lines, key=lambda l: np.mean([it['y_center'] for it in l]))

    # Sort items inside each line left-to-right by X-coordinate
    ordered_text_lines = []
    for line in lines:
        line_sorted = sorted(line, key=lambda it: it['x_min'])
        line_str = " ".join([it['text'] for it in line_sorted])
        ordered_text_lines.append(line_str)

    return "\n".join(ordered_text_lines)

# 7. Run OCR on image
img_path = "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/dataset/js4.jpg"
img = cv2.imread(img_path)

dt_boxes, rec_res, _ = text_sys(img)

# 8. Print properly ordered text
ordered_text = get_perfect_reading_order(dt_boxes, rec_res)

print(ordered_text)


infit enables both web designes and coders to wonte
twith hn andsfiles parally cnd separately ie 
withouttfacing anycode conflictions .
5.  tthelengthoEthecodebeduces . ias oly . weneed 
to specitythelocationoFthe jisfile .


In [ ]:
!python tools/train.py \
    -c "/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/PaddleOCR/configs/det/custom_det_config.yml" \
    -o Global.checkpoints="/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/output/det_finetuned/latest" \
    -o Eval.loader.batch_size_per_card=1

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/08/25 06:39:11] ppocr INFO: Architecture : 
[2026/08/25 06:39:11] ppocr INFO:     Backbone : 
[2026/08/25 06:39:11] ppocr INFO:         det : True
[2026/08/25 06:39:11] ppocr INFO:         name : PPLCNetV3
[2026/08/25 06:39:11] ppocr INFO:         scale : 0.75
[2026/08/25 06:39:11] ppocr INFO:     Head : 
[2026/08/25 06:39:11] ppocr INFO:         fix_nan : True
[2026/08/25 06:39:11] ppocr INFO:         k : 50
[2026/08/25 06:39:11] ppocr INFO:         name : DBHead
[2026/08/25 06:39:11] ppocr INFO:     Neck : 
[2026/08/25 06:39:11] ppocr INFO:         name : RSEFPN
[2026/08/25 06:39:11] ppocr INFO:         out_channels : 9

In [ ]:
import shutil

# Path to the folder you want to zip
source_path = '/content/drive/MyDrive/Colab Notebooks/finetunehuggingface/images'

# Path where the zip file will be created (without .zip extension)
zip_path = '/content/drive/MyDrive/Colab Notebooks'

# Create the zip archive
shutil.make_archive(zip_path, 'zip', source_path)

print("✅ Zipping complete! Archive saved at:", zip_path + ".zip")


✅ Zipping complete! Archive saved at: /content/drive/MyDrive/Colab Notebooks.zip


In [ ]:
import zipfile
import os

# Path to your zip file
zip_path = '/content/drive/MyDrive/Colab Notebooks/datasetpothole/pothole/archive (1).zip'

# Destination folder where you want to extract
extract_path = '/content/drive/MyDrive/Colab Notebooks/datasetpothole/pothole'

# Make sure the destination exists
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipping complete! Files are in:", extract_path)

✅ Unzipping complete! Files are in: /content/drive/MyDrive/Colab Notebooks/datasetpothole/pothole
